# Week 3 - Day 2: Long-term memory

> **Student notebook.** A TODO cell with hints, then a check that prints `PASS` or `NOT YET`.

Yesterday's memory lived in **one thread**: a new conversation started empty. Today the agent remembers a student **across conversations**, like a teacher who remembers what you found hard last week.

| | Short-term memory (Day 1) | Long-term memory (today) |
|---|---|---|
| Where | the checkpointer | a **store** |
| What | the messages of one conversation | facts you choose to keep |
| Lasts | one thread | every thread, for that user |

Today:
1. **The store**: save and read facts, with one **namespace** per student.
2. **Memory tools**: the agent saves and recalls facts itself, with `ToolRuntime`.
3. **A memory agent** that remembers you in a new thread, and never mixes up two students.
4. **A dynamic prompt**: the agent's system prompt is built from your memories before every call.

The code follows the LangChain docs: **[Long-term memory](https://docs.langchain.com/oss/python/langchain/long-term-memory)** and **[Context engineering](https://docs.langchain.com/oss/python/langchain/context-engineering)**.

## Setup
Today you need **one** free key:
- **Groq** (console.groq.com -> API Keys), saved as `GROQ_API_KEY`

**On Google Colab:** click the key icon (Secrets) on the left, add the key, and switch **Notebook access** on. **On your computer:** put it in a `.env` file.

> **Tips.** The free Groq plan allows about 8,000 tokens a minute. If a cell pauses, it is waiting for the limit. If a cell ends with `RateLimitError`, wait a minute and run it again. If a check ends with `NameError`, you have not run the TODO cell above it yet.

In [ ]:
%pip install -qU langchain langgraph langchain-groq langchain-tavily python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()                                                   # on your computer: reads the keys from a .env file
try:
    from google.colab import userdata                           # on Google Colab: reads the keys from Secrets
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    pass

print("GROQ_API_KEY found:", "GROQ_API_KEY" in os.environ)

In [ ]:
from langchain.chat_models import init_chat_model

# One model for the whole week. max_retries=10: when the free plan's speed limit is reached, wait and try again.
llm = init_chat_model("groq:openai/gpt-oss-120b", temperature=0, max_retries=10)

from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

---
# Part 1 - The store

A **store** keeps small pieces of data (Python dictionaries). Each piece has:
- a **namespace**: a tuple that works like a folder, for example `("ram", "memories")`,
- a **key**: the name of the piece inside the folder,
- a **value**: the dictionary itself.

| Method | What it does |
|---|---|
| `store.put(namespace, key, value)` | save (or replace) one piece |
| `store.get(namespace, key)` | read one piece by its key |
| `store.search(namespace)` | list every piece in that folder |

One namespace **per student** keeps their memories apart.

In [ ]:
store = InMemoryStore()
store.put(("ram", "memories"), "food", {"fact": "Ram is vegetarian"})            # namespace, key, value
store.put(("ram", "memories"), "level", {"fact": "Ram is new to Python"})
store.put(("sita", "memories"), "food", {"fact": "Sita likes spicy food"})

In [ ]:
# CHECK 1: read one piece by its key, then list a folder.
item = store.get(("ram", "memories"), "food")
print("get   :", item.value)

for item in store.search(("ram", "memories")):
    print("search:", item.key, "->", item.value["fact"])

ram_count = len(store.search(("ram", "memories")))
sita_count = len(store.search(("sita", "memories")))
if ram_count == 2 and sita_count == 1:
    print("PASS: Ram has 2 memories and Sita has 1, in separate namespaces.")
else:
    print("NOT YET: Ram should have 2 memories and Sita 1. Check the namespaces in store.put.")

---
# Part 2 - Memory tools

The agent should save and recall facts **by itself**. It does that with two tools. A tool reaches the store through **`ToolRuntime`**: an extra argument that LangChain fills in. The model never sees it, so it cannot choose another student's folder:
- `runtime.store` is the store,
- `runtime.context.user_id` is the student who is talking. **Your code** gives it when you call the agent: `context=Context(user_id="ram")`.

`Context` is a small `dataclass`: a class that only holds named values.

In [ ]:
@dataclass
class Context:
    user_id: str

@tool
def save_memory(fact: str, runtime: ToolRuntime[Context]) -> str:
    """Save one short fact about the student, for example how they like to learn or a date to remember."""
    runtime.store.put((runtime.context.user_id, "memories"), fact, {"fact": fact})      # the fact is also the key
    return "Saved: " + fact

@tool
def recall_memories(runtime: ToolRuntime[Context]) -> str:
    """List everything saved about the student."""
    items = runtime.store.search((runtime.context.user_id, "memories"))
    if not items:
        return "Nothing saved about this student yet."
    facts = []
    for item in items:
        facts.append(item.value["fact"])
    return "\n".join(facts)

In [ ]:
# CHECK 2: the model sees only "fact", never "runtime".
print("save_memory arguments    :", list(save_memory.args))
print("recall_memories arguments:", list(recall_memories.args))

if list(save_memory.args) == ["fact"] and list(recall_memories.args) == []:
    print("PASS: the model cannot see or choose the runtime, so it cannot pick another student's folder.")
else:
    print("NOT YET: the only argument the model sees should be fact. Check that runtime is typed ToolRuntime[Context].")

---
# Part 3 - A memory agent

`create_agent` gets three more arguments:
- `store=store`: the long-term memory,
- `context_schema=Context`: the shape of the context,
- `checkpointer=InMemorySaver()`: the short-term memory of Day 1.

Then you talk to it as **Ram** in one thread, as **Ram** again in a **new** thread, and as **Sita**.

In [ ]:
memory_agent = create_agent(
    llm,
    tools=[save_memory, recall_memories],
    system_prompt="You are a study assistant. When the student tells you something about themselves, save it with save_memory. "
                  "Before you answer a question about the student, use recall_memories.",
    store=store,                    # long-term memory
    context_schema=Context,         # who is talking
    checkpointer=InMemorySaver(),   # short-term memory
)

In [ ]:
# Given: three conversations.
ram = Context(user_id="ram")
sita = Context(user_id="sita")

first = memory_agent.invoke({"messages": "Please remember: I learn best with short examples, and my exam is on 1 December."},
                            {"configurable": {"thread_id": "ram-1"}}, context=ram)
print("Ram, thread 1 :", first["messages"][-1].content)

later = memory_agent.invoke({"messages": "What do you know about me, and when is my exam?"},
                            {"configurable": {"thread_id": "ram-2"}}, context=ram)          # a NEW thread
print("Ram, thread 2 :", later["messages"][-1].content)

other = memory_agent.invoke({"messages": "When is my exam?"},
                            {"configurable": {"thread_id": "sita-1"}}, context=sita)
print("Sita, thread 1:", other["messages"][-1].content)

print()
for item in store.search(("ram", "memories")):
    print("Ram's memory:", item.value["fact"])

In [ ]:
# CHECK 3
ram_facts = ""
for item in store.search(("ram", "memories")):
    ram_facts = ram_facts + item.value["fact"].lower()

if "december" in ram_facts:
    print("PASS: the agent saved the exam date in Ram's namespace.")
else:
    print("NOT YET: the exam date should be saved for Ram. Check save_memory and the system prompt.")
if "december" in later["messages"][-1].content.lower():
    print("PASS: in a new thread, the agent still knows Ram's exam date.")
else:
    print("NOT YET: in thread ram-2 the agent should find the date with recall_memories.")
if "december" not in other["messages"][-1].content.lower():
    print("PASS: Sita never sees Ram's memories.")
else:
    print("NOT YET: Sita got Ram's date! Check that the namespace uses runtime.context.user_id.")

---
# Part 4 - A dynamic prompt

With tools, the agent must **decide** to recall. Sometimes it forgets to. A **dynamic prompt** puts the memories in the system prompt **before every model call**, so the model always has them.

`@dynamic_prompt` turns a function into middleware. The function gets the **request** of the model call and returns the system prompt. `request.runtime.store` and `request.runtime.context` work like in the tools.

In [ ]:
from langchain.agents.middleware import dynamic_prompt

@dynamic_prompt
def personal_prompt(request):
    items = request.runtime.store.search((request.runtime.context.user_id, "memories"))
    facts = ""
    for item in items:
        facts = facts + "- " + item.value["fact"] + "\n"
    return "You are a study assistant. What you know about this student:\n" + facts      # built again before every model call

prompt_agent = create_agent(llm, tools=[save_memory], middleware=[personal_prompt],
                            store=store, context_schema=Context, checkpointer=InMemorySaver())

In [ ]:
# CHECK 4: a new thread, no recall tool. The memories come in through the prompt.
answer = prompt_agent.invoke({"messages": "When is my exam? Answer in one sentence."},
                             {"configurable": {"thread_id": "ram-3"}}, context=ram)
print(answer["messages"][-1].content)

called_tool = False
for message in answer["messages"]:
    if message.type == "tool":
        called_tool = True

if "december" in answer["messages"][-1].content.lower() and not called_tool:
    print("PASS: the agent knew the date from its system prompt, without calling a tool.")
else:
    print("NOT YET: the answer should give 1 December. Check personal_prompt and middleware=[personal_prompt].")

In [ ]:
tools_or_prompt = ("With a few memories, the prompt is best, because the model always has them and does not have to remember to call a tool. "
                   "With thousands of memories the prompt would be far too long, so a tool that finds only the memories you need is better.")

In [ ]:
# CHECK 5
if len(tools_or_prompt.split()) >= 20:
    print("PASS")
else:
    print("NOT YET: write two sentences.")

---
# Today you learned
- A **store** keeps facts across conversations. A **namespace** per student keeps them apart.
- Tools reach the store through **`ToolRuntime`**; the user id comes from **your code** (`context=`), never from the model.
- A **memory agent** = tools + `store=` + `context_schema=` + a checkpointer.
- A **dynamic prompt** puts the memories in the system prompt before every call.

**Next class: agentic RAG.** The agent searches **documents**: the notes of this course.